In [63]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain.agents.middleware import wrap_model_call
import subprocess
from pathlib import Path
import sys
from typing import Optional

load_dotenv()

True

In [3]:
embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 939.35it/s]


In [4]:
django_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="django_docs",
    embedding_function=embeddings
)

python_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="python_scripts",
    embedding_function=embeddings
)

In [5]:
django_db= django_vectorstore.get()
python_db=python_vectorstore.get()

django_splits=[
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(django_db["documents"], django_db["metadatas"])
]

python_splits = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(python_db["documents"], python_db["metadatas"])
]

django_retriever= django_vectorstore.as_retriever(search_kwargs={"k":4})
python_retriever= python_vectorstore.as_retriever(search_kwargs={"k":4})


all_splits= django_splits + python_splits
print(f"Loaded {len(all_splits)} total splits ({len(django_splits)} Django docs + {len(python_splits)} Python codebase).")
bm25_retriever= BM25Retriever.from_documents(all_splits)


Loaded 6361 total splits (5110 Django docs + 1251 Python codebase).


In [6]:
bm25_retriever.k=8

In [7]:
#Creating hybrid retriever
hybrid_retriever= EnsembleRetriever(
    retrievers=[django_retriever, python_retriever, bm25_retriever],
    weights=[0.4,0.3,0.3]#40% django sementic search, 30% python sementic and keyword
    
)

In [8]:
#Reranker block of code
reranker= CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1571.11it/s]


## Generation Tools

In [70]:
DEFAULT_DIR = r"C:\Users\ZENOID\Desktop\Home\home\self_made.projects\AI(created by me) generated projects"

@tool
def retrieve_django_content(query: str) -> str:
    """
    Search the Django knowledge base for documentation, code examples,
    debugging information, and practical examples.
    Use this tool to gain more context and information needed to answer the users question
    """
    print("Using retriever......")
    docs = hybrid_retriever.invoke(query)

    print("Reranking......")
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )
    top_docs = scored_docs[:3]

    return "\n\n".join(doc.page_content for doc, score in top_docs)


def _get_bin_dir(env_path: Path) -> Path:
    """Returns the folder inside a venv where executables live (differs by OS)."""
    return env_path / "Scripts" if sys.platform == "win32" else env_path / "bin"


def _exe(bin_dir: Path, name: str) -> Path:
    """Adds .exe to the executable name only on Windows."""
    return bin_dir / (f"{name}.exe" if sys.platform == "win32" else name)


@tool
def create_virtualenv(project_name: str, directory: Optional[str]= None) -> str:
    """Runs a command to create a new virtual environment with the given name in the specified directory."""
    if not directory or directory.strip() in (".", ""):
        directory = DEFAULT_DIR
    print("Creating virtual environent......")
    target_path = Path(directory) / project_name
    target_path.mkdir(parents=True, exist_ok=True)
    if (target_path / "env").exists():
        return f"Virtual environment already exists in {directory}"
    result = subprocess.run(
        [sys.executable, "-m", "venv", "env"],
        cwd=target_path, capture_output=True, text=True
    )
    if result.returncode != 0:
        return f"Failed to create virtual environment: {result.stderr}"
    print(f"Created virtual environment env in {directory}")
    return f"Created virtual environment env in {directory}"


def _install_django(project_name: str, directory: Optional[str]= None) -> str:
    """Runs a command to install Django in the virtual environment."""
    if not directory or directory.strip() in (".", ""):
        directory = DEFAULT_DIR
    print("Installing Django......")
    target_path = Path(directory) / project_name
    pip_exe = _exe(_get_bin_dir(target_path / "env"), "pip")
    result = subprocess.run(
        [str(pip_exe), "install", "django"],
        cwd=target_path, capture_output=True, text=True
    )
    if result.returncode != 0:
        return f"Failed to install Django: {result.stderr}"
    return "Installed Django in virtual environment env"


@tool
def create_django_project(project_name: str, directory: Optional[str] = None) -> str:
    """Runs a command to create a new Django project with the given name in the specified directory."""
    if not directory or directory.strip() in (".", ""):
        directory = DEFAULT_DIR
    print("Creating Django project......")
    target_path = Path(directory) / project_name
    target_path.mkdir(parents=True, exist_ok=True)
    bin_dir = _get_bin_dir(target_path / "env")
    pip_exe = _exe(bin_dir, "pip")
    django_admin = _exe(bin_dir, "django-admin")

    check = subprocess.run([str(pip_exe), "list"], capture_output=True, text=True)
    if "django" not in check.stdout.lower():
        install_result = _install_django(project_name, directory)
        if "Failed" in install_result:
            return install_result

    result = subprocess.run(
        [str(django_admin), "startproject", project_name, "."],
        cwd=target_path, capture_output=True, text=True
    )
    if result.returncode != 0:
        return f"Failed to create Django project: {result.stderr}"
    return f"Created Django project {project_name} in {directory}"


_active_server_process = None

@tool
def run_django_server(project_name: str, directory: Optional[str]=None) -> str:
    """Runs a command to start the Django development server asynchronously."""
    if not directory or directory.strip() in (".", ""):
        directory = DEFAULT_DIR
    print("Running Django server......")
    global _active_server_process

    target_path = Path(directory) / project_name
    if not (target_path / "manage.py").exists():
        target_path = Path(directory)

    manage_py = target_path / "manage.py"
    if not manage_py.exists():
        return f"Could not find manage.py in {target_path} — make sure the project was created first."

    python_exe = _exe(_get_bin_dir(target_path.parent / "env"), "python")
    if not python_exe.exists():
        python_exe = _exe(_get_bin_dir(target_path / "env"), "python")

    if _active_server_process and _active_server_process.poll() is None:
        _active_server_process.terminate()

    _active_server_process = subprocess.Popen(
    [str(python_exe), "manage.py", "runserver", "--noreload"],
    cwd=target_path
    )
    return f"Django server initiated in {target_path}"


@tool
def stop_django_server() -> str:
    """Stops the currently running Django development server task."""
    print("Stopping Django server......")
    global _active_server_process

    if _active_server_process and _active_server_process.poll() is None:
        _active_server_process.terminate()
        try:
            _active_server_process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            _active_server_process.kill()
        _active_server_process = None
        return "The Django server has been successfully stopped."

    return "No active Django server is currently running."

@tool 
def create_django_app(name: str, project_name: str, directory: Optional[str] = None) -> str:
    """Run a command to create a new Django app with the given name inside the specified Django project."""
    if not directory or directory.strip() in (".", ""):
        directory = DEFAULT_DIR
    print("Creating Django app......")
    target_path = Path(directory) / project_name
    bin_dir = _get_bin_dir(target_path / "env")
    pip_exe = _exe(bin_dir, "pip")
    django_admin = _exe(bin_dir, "django-admin")

    check = subprocess.run([str(pip_exe), "list"], capture_output=True, text=True)
    if "django" not in check.stdout.lower():
        install_result = _install_django(project_name, directory)
        if "Failed" in install_result:
            return install_result

    result = subprocess.run(
        [str(django_admin), "startapp", name],
        cwd=target_path, capture_output=True, text=True
    )
    if result.returncode != 0:
        return f"Failed to create Django app: {result.stderr}"
    return f"Created Django app {name} in {target_path}"

## File Editor Tools
Prompt/Task -> Search -> Create/Read -> Edit -> Write

In [71]:
#Constructing Agent and memory
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
memory=MemorySaver()



In [72]:
#Deleting memory history/cache
# memory.storage.pop("test-8", None)

In [73]:
#Trimming users messages to preserve context window
@wrap_model_call
def limit_history(request, handler):
    messages = request.state["messages"]

    recent_messages = messages[-2:]

    request.state["messages"] = recent_messages

    return handler(request)

In [74]:
agent= create_agent(
    model=llm,
    tools=[retrieve_django_content, create_virtualenv, create_django_project, run_django_server, stop_django_server,create_django_app ],
    system_prompt = r"""
You are a Django engineering assistant with two jobs: reviewing Django code, and setting up/running Django projects using the tools available to you.

You have two categories of tools:

1. KNOWLEDGE TOOL — retrieve_django_content
   Call this when the user asks a question, wants a code review, or wants an explanation of a Django concept, pattern, or error — anything where getting it right depends on accurate technical detail.
   Do NOT call this for simple action requests that don't require technical judgment — e.g. "create a project called X," "create an app called Y," "start the server," "stop the server." These just need the right tool called correctly, not a documentation lookup.
   If a request mixes both (e.g. "review this code, then create a project"), call retrieve_django_content only for the review part.

2. ACTION TOOLS — these perform real actions on the user's machine:
   - create_virtualenv(directory): creates a virtual environment named "env" inside the given directory
   - create_django_project(name, directory): creates a new Django project (installs Django first if missing)
   - create_django_app(name, project_name, directory): creates a new Django app inside an existing project
   - run_django_server(project_name, directory): starts the Django dev server in the background
   - stop_django_server(): stops the currently running dev server

   DIRECTORY RULE:
   - If the user's request doesn't specify a directory, leave the `directory` parameter out of the tool call entirely — do not fill it with "." or the current workspace path. The tool itself will fall back to the correct default folder.
   - Never guess or invent a directory path on your own when none is given.

STRICT ORDER OF OPERATIONS (for action requests):
1. create_virtualenv → 2. create_django_project → 3. create_django_app (only if an app was requested) → 4. run_django_server
   Skip a step only if the user says it already exists.

RULES FOR ACTION TOOLS:
- If the user asks you to create, set up, scaffold, start, or run a Django project or app — use the tools. Do not just describe the steps in text; call the tools.
- Infer sensible defaults if the user is vague (e.g. project name "myproject") and state the assumption in one short sentence before acting.
- After calling a tool, report what actually happened in one or two sentences — summarize, don't paste the tool's raw return string verbatim.
- If a tool call fails or the server doesn't start, say so plainly and suggest the likely cause — do not pretend it succeeded.
- Never call run_django_server without first confirming a project actually exists at that path.
- Never call create_django_app without first confirming the target project actually exists.
- Only call stop_django_server if the user asks to stop, restart, or if you are about to start a new server and one may already be running.
- If the user only asks you to explain what steps you'd take, describe them in words — do not call any tool.

RULES FOR CODE REVIEW / KNOWLEDGE ANSWERS:
- Answer using the context retrieved from retrieve_django_content. If it doesn't return anything relevant, say "I don't have this information in my knowledge base" rather than guessing.
- Keep review answers direct: lead with the answer, then 1-3 sentences of essential context.
- Use bullet points only when the user asks for a list of issues/items.
- Don't narrate your retrieval or tool-selection process to the user (no "I searched the knowledge base..." or "I'm now calling retrieve_django_content...") — just do it and report the outcome.
- retrieve_django_content only covers Django documentation and technical content — never call it for questions about this conversation's own history or what was previously created; answer those from the conversation itself, or say plainly if you don't have that information.

GENERAL:
- Be concise. No filler like "Based on the above" or "In conclusion."
- If a request is ambiguous between "explain how to do X" and "do X for me," default to doing it if action tools are relevant — ask only if the ambiguity would cause you to act on the wrong project/directory.
""",
    middleware=[limit_history],
    checkpointer=memory
)

#Memory
config = {"configurable": {"thread_id": "test-12"}}

In [75]:
#Creatin User Interface
while True:
    question= input("Any Question about django: ").strip()
    if question.lower() in ["exit", "quit", "q"]:
        print("See you soon...")
        break
    if not question:
        continue

    result= agent.invoke({
        "messages": [HumanMessage(content=question)]
    }, config=config)

    print("\n Final Result")
    print(result["messages"][-1].content)

Creating virtual environent......
Created virtual environment env in C:\Users\ZENOID\Desktop\Home\home\self_made.projects\AI(created by me) generated projects
Creating Django project......
Installing Django......
Creating Django app......
Running Django server......

 Final Result
The Django project **shopsite** with the **products** app has been created, and the development server is now running. Let me know if you need anything else!
Creating Django app......

 Final Result
The **prices** app has been added to the **shopsite** project. Let me know if you’d like any further setup or changes!

 Final Result
Your **shopsite** project currently contains **2 apps**: `products` and `prices`.

 Final Result
You’re welcome! If you need anything else, just let me know.
See you soon...
